# Chunking Strategy R&D

Goal:
Compare different chunking strategies for insurance policy documents.

Why:
Chunking quality directly affects retrieval quality, embedding quality, and final answer accuracy.

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print("Imports successful")

C:\Users\Vikram\AppData\Local\Temp\ipykernel_25752\3105127256.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\Vikram\insurance_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful


# Chunking Strategy R&D

Objective:
Compare different chunking strategies for insurance policy documents and identify the most suitable approach for the Insurance RAG system.

Strategies:
- Character Chunking
- Recursive Chunking
- Token Chunking

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

import pandas as pd

In [3]:
PDF_PATH = "../data/general/health/SBI_Health_Insurance_Retail.pdf"

loader = PyMuPDFLoader(PDF_PATH)
docs = loader.load()

text = "\n".join([doc.page_content for doc in docs])

print("Total Length:", len(text))
print(text[:1000])

Total Length: 92847
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company Limited
Limited
Limited
Limited
                 
 
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited   
Corporate & Registered Office:   'Natraj', 301, Junction of  Western Express Highway & Andheri - Kurla Road, Andheri (E), Mumbai - 400 069 I 
CIN: U66000MH2009PLC190546 I    Tel.: +91 22 42412000 I  
 www.sbigeneral.in I Logo displayed belongs to State Bank of India and is used 
by SBI General Insurance Co. Ltd. under license I IRDAI Registration Number 144 I  Product Name: Health Insurance Policy Retail. I  
UIN: SBIHLIP11002V021011 I IRDAI Reg No.144 
 
HEALTH INSURANCE POLICY –RETAIL 
 
This Policy is issued to the Insured based on the Proposal and declaration together with any statement, report or 
other document which shall be the 

In [4]:
character_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separator="\n"
)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

token_splitter = TokenTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

In [5]:
character_chunks = character_splitter.split_text(text)
recursive_chunks = recursive_splitter.split_text(text)
token_chunks = token_splitter.split_text(text)

print("Character Chunks:", len(character_chunks))
print("Recursive Chunks:", len(recursive_chunks))
print("Token Chunks:", len(token_chunks))

Character Chunks: 116
Recursive Chunks: 116
Token Chunks: 100


In [6]:
print("="*100)
print("CHARACTER")
print("="*100)
print(character_chunks[0])

print("\n\n")

print("="*100)
print("RECURSIVE")
print("="*100)
print(recursive_chunks[0])

print("\n\n")

print("="*100)
print("TOKEN")
print("="*100)
print(token_chunks[0])

CHARACTER
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company Limited
Limited
Limited
Limited
                 
 
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited   
Corporate & Registered Office:   'Natraj', 301, Junction of  Western Express Highway & Andheri - Kurla Road, Andheri (E), Mumbai - 400 069 I 
CIN: U66000MH2009PLC190546 I    Tel.: +91 22 42412000 I  
 www.sbigeneral.in I Logo displayed belongs to State Bank of India and is used 
by SBI General Insurance Co. Ltd. under license I IRDAI Registration Number 144 I  Product Name: Health Insurance Policy Retail. I  
UIN: SBIHLIP11002V021011 I IRDAI Reg No.144 
 
HEALTH INSURANCE POLICY –RETAIL 
 
This Policy is issued to the Insured based on the Proposal and declaration together with any statement, report or



RECURSIVE
SBI General Insurance Company 
S

In [7]:
results = []

for name, chunks in [
    ("Character", character_chunks),
    ("Recursive", recursive_chunks),
    ("Token", token_chunks)
]:
    
    avg_len = sum(len(c) for c in chunks) / len(chunks)

    results.append({
        "Strategy": name,
        "Chunks": len(chunks),
        "Average Length": round(avg_len, 2),
        "Min Length": min(len(c) for c in chunks),
        "Max Length": max(len(c) for c in chunks)
    })

pd.DataFrame(results)

,Strategy,Chunks,Average Length,Min Length,Max Length
0,Character,116,955.51,770,998
1,Recursive,116,955.05,770,998
2,Token,100,1109.90,630,1391


In [8]:
for i in range(3):

    print("\n" + "="*100)
    print(f"RECURSIVE CHUNK {i}")
    print("="*100)

    print(recursive_chunks[i])


RECURSIVE CHUNK 0
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company Limited
Limited
Limited
Limited
                 
 
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited   
Corporate & Registered Office:   'Natraj', 301, Junction of  Western Express Highway & Andheri - Kurla Road, Andheri (E), Mumbai - 400 069 I 
CIN: U66000MH2009PLC190546 I    Tel.: +91 22 42412000 I  
 www.sbigeneral.in I Logo displayed belongs to State Bank of India and is used 
by SBI General Insurance Co. Ltd. under license I IRDAI Registration Number 144 I  Product Name: Health Insurance Policy Retail. I  
UIN: SBIHLIP11002V021011 I IRDAI Reg No.144 
 
HEALTH INSURANCE POLICY –RETAIL 
 
This Policy is issued to the Insured based on the Proposal and declaration together with any statement, report or

RECURSIVE CHUNK 1
UIN: SBIHLIP11002

In [9]:
for i in range(3):

    print("\n" + "="*100)
    print(f"TOKEN CHUNK {i}")
    print("="*100)

    print(token_chunks[i])


TOKEN CHUNK 0
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company Limited
Limited
Limited
Limited
                 
 
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited   
Corporate & Registered Office:   'Natraj', 301, Junction of  Western Express Highway & Andheri - Kurla Road, Andheri (E), Mumbai - 400 069 I 
CIN: U66000MH2009PLC190546 I    Tel.: +91 22 42412000 I  
 www.sbigeneral.in I Logo displayed belongs to State Bank of India and is used 
by SBI General Insurance Co. Ltd. under license I IRDAI Registration Number 144 I  Product Name: Health Insurance Policy Retail. I  
UIN: SBIHLIP11002V021011 I IRDAI Reg No.144 
 
HEALTH INSURANCE POLICY –RETAIL 
 
This Policy is issued to the Insured based on the Proposal and declaration together with any statement, report or 
other document which shall be the basis

In [10]:
import re

def clean_text(text):
    text = text.replace("\uf0b7", "•")
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"Page\s+\d+\s+of\s+\d+", "", text, flags=re.IGNORECASE)
    return text.strip()

In [11]:
cleaned_text = clean_text(text)

character_chunks = character_splitter.split_text(cleaned_text)
recursive_chunks = recursive_splitter.split_text(cleaned_text)
token_chunks = token_splitter.split_text(cleaned_text)

print("Character Chunks:", len(character_chunks))
print("Recursive Chunks:", len(recursive_chunks))
print("Token Chunks:", len(token_chunks))

Character Chunks: 114
Recursive Chunks: 129
Token Chunks: 95


In [12]:
results = []

for name, chunks in [
    ("Character Cleaned", character_chunks),
    ("Recursive Cleaned", recursive_chunks),
    ("Token Cleaned", token_chunks)
]:
    avg_len = sum(len(c) for c in chunks) / len(chunks)

    results.append({
        "Strategy": name,
        "Chunks": len(chunks),
        "Average Length": round(avg_len, 2),
        "Min Length": min(len(c) for c in chunks),
        "Max Length": max(len(c) for c in chunks)
    })

pd.DataFrame(results)

,Strategy,Chunks,Average Length,Min Length,Max Length
0,Character Cleaned,114,953.76,689,998
1,Recursive Cleaned,129,756.09,18,997
2,Token Cleaned,95,1153.82,647,1428


In [13]:
for i in range(2):
    print("\n" + "="*100)
    print(f"RECURSIVE CLEANED CHUNK {i}")
    print("="*100)
    print(recursive_chunks[i])


RECURSIVE CLEANED CHUNK 0
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company Limited
Limited
Limited
Limited

SBI General Insurance Company Limited 
SBI General Insurance Company Limited 
SBI General Insurance Company Limited 
SBI General Insurance Company Limited 
Corporate & Registered Office: 'Natraj', 301, Junction of Western Express Highway & Andheri - Kurla Road, Andheri (E), Mumbai - 400 069 I 
CIN: U66000MH2009PLC190546 I Tel.: +91 22 42412000 I 
 www.sbigeneral.in I Logo displayed belongs to State Bank of India and is used 
by SBI General Insurance Co. Ltd. under license I IRDAI Registration Number 144 I Product Name: Health Insurance Policy Retail. I 
UIN: SBIHLIP11002V021011 I IRDAI Reg No.144 

HEALTH INSURANCE POLICY –RETAIL

RECURSIVE CLEANED CHUNK 1
HEALTH INSURANCE POLICY –RETAIL 

This Policy is issued to the Insured based on the Proposal and declaration together with any statement, report or 
oth

In [14]:
small_chunks = [
    (i, chunk) for i, chunk in enumerate(recursive_chunks)
    if len(chunk) < 100
]

print("Small chunks count:", len(small_chunks))

for i, chunk in small_chunks[:10]:
    print("\nIndex:", i)
    print("Length:", len(chunk))
    print(repr(chunk))

Small chunks count: 2

Index: 62
Length: 18
'General Conditions'

Index: 68
Length: 21
'f) \nClaims Procedures'


In [15]:
def remove_small_chunks(chunks, min_length=100):
    return [chunk for chunk in chunks if len(chunk.strip()) >= min_length]

In [16]:
recursive_chunks_filtered = remove_small_chunks(recursive_chunks)

print("Before:", len(recursive_chunks))
print("After:", len(recursive_chunks_filtered))

Before: 129
After: 127


In [17]:
recursive_chunks_final = recursive_chunks_filtered

In [18]:
final_results = []

for name, chunks in [
    ("Character Cleaned", character_chunks),
    ("Recursive Cleaned + Filtered", recursive_chunks_final),
    ("Token Cleaned", token_chunks)
]:
    avg_len = sum(len(c) for c in chunks) / len(chunks)

    final_results.append({
        "Strategy": name,
        "Chunks": len(chunks),
        "Average Length": round(avg_len, 2),
        "Min Length": min(len(c) for c in chunks),
        "Max Length": max(len(c) for c in chunks)
    })

pd.DataFrame(final_results)

,Strategy,Chunks,Average Length,Min Length,Max Length
0,Character Cleaned,114,953.76,689,998
1,Recursive Cleaned + Filtered,127,767.69,110,997
2,Token Cleaned,95,1153.82,647,1428


# Final Chunking Decision

Strategies Evaluated:
- Character Chunking
- Recursive Chunking
- Token Chunking

Result:
Recursive Chunking provided the best balance between chunk size, context preservation, and retrieval suitability.

Selected Configuration:

chunk_size = 1000
chunk_overlap = 200

Additional Rule:
Remove chunks smaller than 100 characters.

Reason:
Insurance policies contain structured sections such as benefits, exclusions, waiting periods, and claim conditions. Recursive chunking preserves these boundaries better than character-based splitting while avoiding excessively large chunks generated by token chunking.